# Phase 9 — Les cases vides

## Objectifs

- Identifier les trois colonnes les plus incomplètes.
- Comparer la proportion de canulars entre les relevés ayant une valeur manquante
  et ceux dont la valeur est renseignée.
- Choisir une méthode de traitement qui conserve l'information selon laquelle la
  donnée était initialement manquante.
- Préparer un modèle pouvant distinguer une valeur réellement observée d'une
  valeur imputée.

## 1. Imports des bibliothèques

In [1]:
from pathlib import Path
import csv
import re

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

## 2. Chemins, colonnes et paramètres

Les résultats de cette phase seront enregistrés dans :

```text
outputs/phase_9_valeurs_manquantes/
```

In [2]:
DATA_PATH = Path("../data/releves_klaxo3.csv")

OUTPUT_DIR = Path("../outputs")
PHASE9_DIR = OUTPUT_DIR / "phase_9_valeurs_manquantes"
PHASE9_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

MOTS_CLES_CANULAR = [
    "hoax",
    "fake",
    "prank",
    "joke",
    "not real",
    "made up",
    "fraud",
]

TEST_SIZE = 0.20
RANDOM_STATE = 42

## 3. Chargement robuste du fichier

Seules les lignes contenant exactement 11 champs sont chargées dans le DataFrame
principal. Les lignes mal structurées restent isolées afin que les données ne
disparaissent pas silencieusement.

In [3]:
lignes_valides = []
lignes_problemes = []

with open(
    DATA_PATH,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append(
                {
                    "numero_ligne": numero_ligne,
                    "nb_champs": len(row),
                    "contenu": row,
                }
            )

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Nombre de lignes chargées : {len(df)}")
print(f"Nombre de lignes isolées : {len(lignes_problemes)}")

Nombre de lignes chargées : 88679
Nombre de lignes isolées : 196


## 4. Conversion des colonnes numériques et des dates

Les conversions sont effectuées sans supprimer de lignes. Une valeur invalide ou
absente devient une valeur manquante (`NaN` ou `NaT`).

In [4]:
for col in ["duration_seconds", "latitude", "longitude"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

for col in ["datetime", "date_posted"]:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration_seconds             float64
duration_hours_min            object
comments                      object
date_posted           datetime64[ns]
latitude                     float64
longitude                    float64
dtype: object

## 5. Création de la cible artificielle `is_hoax`

La cible est reconstruite avec la même règle que dans les phases précédentes.
Un relevé est marqué comme canular lorsque son commentaire contient au moins un
mot-clé de la liste définie.

In [5]:
pattern_canular = "|".join(
    re.escape(mot)
    for mot in MOTS_CLES_CANULAR
)

df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(
        pattern_canular,
        regex=True,
        na=False,
    )
    .astype(int)
)

print(df["is_hoax"].value_counts())

is_hoax
0    87810
1      869
Name: count, dtype: int64


## 6. Uniformisation des valeurs manquantes textuelles

Dans les colonnes textuelles, une chaîne vide ou contenant uniquement des espaces
doit être considérée comme une valeur manquante.

Cette étape permet de comparer équitablement les valeurs manquantes de toutes
les colonnes.

In [6]:
colonnes_textuelles = [
    "city",
    "state",
    "country",
    "shape",
    "duration_hours_min",
    "comments",
]

for col in colonnes_textuelles:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

df[colonnes_textuelles].isna().sum()

city                      0
state                  7409
country               12365
shape                  2922
duration_hours_min     3017
comments                 35
dtype: int64

## 7. Comptage des valeurs manquantes

Le nombre et le pourcentage de valeurs manquantes sont calculés pour chaque
colonne. Les trois colonnes les plus incomplètes seront étudiées ensuite.

In [7]:
resume_manquants = pd.DataFrame(
    {
        "nombre_manquants": df.isna().sum(),
        "proportion_manquants": df.isna().mean(),
    }
)

resume_manquants["proportion_manquants_pct"] = (
    resume_manquants["proportion_manquants"]
    .mul(100)
    .round(2)
)

resume_manquants = resume_manquants.sort_values(
    "nombre_manquants",
    ascending=False,
)

resume_manquants

,nombre_manquants,proportion_manquants,proportion_manquants_pct
country,12365,0.139435,13.94
state,7409,0.083549,8.35
duration_hours_min,3017,0.034022,3.40
shape,2922,0.032950,3.30
datetime,1220,0.013757,1.38
comments,35,0.000395,0.04
duration_seconds,5,0.000056,0.01
latitude,1,0.000011,0.00
city,0,0.000000,0.00
date_posted,0,0.000000,0.00


## 8. Sélection des trois colonnes les plus trouées

Les trois colonnes avec le plus grand nombre de valeurs manquantes sont
retenues pour l'analyse demandée dans cette phase.

In [8]:
trois_colonnes_plus_trouees = (
    resume_manquants
    .head(3)
    .index
    .tolist()
)

print("Les trois colonnes les plus trouées sont :")
print(trois_colonnes_plus_trouees)

Les trois colonnes les plus trouées sont :
['country', 'state', 'duration_hours_min']


## 9. Comparaison de la proportion de canulars

Pour chaque colonne sélectionnée, deux proportions sont calculées :

- la proportion de canulars parmi les relevés où la colonne est manquante ;
- la proportion de canulars parmi les relevés où la colonne est renseignée.

Cette mesure est effectuée avant toute imputation afin de ne pas effacer le
signal potentiellement contenu dans l'absence d'information.

In [9]:
resultats_manquants = []

for colonne in trois_colonnes_plus_trouees:
    masque_manquant = df[colonne].isna()
    masque_present = df[colonne].notna()

    resultats_manquants.append(
        {
            "colonne": colonne,
            "nombre_manquants": int(masque_manquant.sum()),
            "nombre_presents": int(masque_present.sum()),
            "canulars_parmi_manquants": int(
                df.loc[masque_manquant, "is_hoax"].sum()
            ),
            "canulars_parmi_presents": int(
                df.loc[masque_present, "is_hoax"].sum()
            ),
            "proportion_canulars_manquants": (
                df.loc[masque_manquant, "is_hoax"].mean()
            ),
            "proportion_canulars_presents": (
                df.loc[masque_present, "is_hoax"].mean()
            ),
        }
    )

df_resultats_manquants = pd.DataFrame(
    resultats_manquants
)

df_resultats_manquants

,colonne,nombre_manquants,nombre_presents,canulars_parmi_manquants,canulars_parmi_presents,proportion_canulars_manquants,proportion_canulars_presents
0,country,12365,76314,150,719,0.012131,0.009422
1,state,7409,81270,103,766,0.013902,0.009425
2,duration_hours_min,3017,85662,79,790,0.026185,0.009222


## 10. Mise en forme des résultats

Les proportions sont converties en pourcentages pour faciliter leur lecture et
leur ajout au rapport.

In [10]:
df_resultats_manquants_affichage = (
    df_resultats_manquants
    .copy()
)

df_resultats_manquants_affichage[
    "proportion_canulars_manquants_pct"
] = (
    df_resultats_manquants_affichage[
        "proportion_canulars_manquants"
    ]
    .mul(100)
    .round(3)
)

df_resultats_manquants_affichage[
    "proportion_canulars_presents_pct"
] = (
    df_resultats_manquants_affichage[
        "proportion_canulars_presents"
    ]
    .mul(100)
    .round(3)
)

df_resultats_manquants_affichage[
    [
        "colonne",
        "nombre_manquants",
        "nombre_presents",
        "proportion_canulars_manquants_pct",
        "proportion_canulars_presents_pct",
    ]
]

,colonne,nombre_manquants,nombre_presents,proportion_canulars_manquants_pct,proportion_canulars_presents_pct
0,country,12365,76314,1.213,0.942
1,state,7409,81270,1.390,0.943
2,duration_hours_min,3017,85662,2.618,0.922


## 11. Interprétation de la différence observée

Cette cellule calcule la différence entre la proportion de canulars parmi les
lignes incomplètes et celle des lignes complètes.

Une différence non nulle indique que le fait qu'une valeur soit manquante peut
être informatif pour prédire la cible.

In [11]:
df_resultats_manquants_affichage[
    "ecart_points_pourcentage"
] = (
    df_resultats_manquants_affichage[
        "proportion_canulars_manquants_pct"
    ]
    - df_resultats_manquants_affichage[
        "proportion_canulars_presents_pct"
    ]
).round(3)

df_resultats_manquants_affichage[
    [
        "colonne",
        "proportion_canulars_manquants_pct",
        "proportion_canulars_presents_pct",
        "ecart_points_pourcentage",
    ]
]

,colonne,proportion_canulars_manquants_pct,proportion_canulars_presents_pct,ecart_points_pourcentage
0,country,1.213,0.942,0.271
1,state,1.390,0.943,0.447
2,duration_hours_min,2.618,0.922,1.696


## 12. Préparation des variables temporelles et textuelles

Les informations créées ici sont les mêmes que celles utilisées dans les phases
précédentes. Le texte des commentaires reste exclu des entrées du modèle, car
il sert à construire la cible `is_hoax`.

In [12]:
df["observation_year"] = df["datetime"].dt.year
df["observation_month"] = df["datetime"].dt.month
df["observation_hour"] = df["datetime"].dt.hour

df["text_features_without_leakage"] = (
    "city " + df["city"].fillna("<MANQUANT>").astype(str)
    + " state " + df["state"].fillna("<MANQUANT>").astype(str)
    + " country " + df["country"].fillna("<MANQUANT>").astype(str)
    + " shape " + df["shape"].fillna("<MANQUANT>").astype(str)
)

## 13. Choix du traitement des valeurs manquantes

Le traitement retenu est le suivant :

- les variables numériques sont imputées par leur médiane ;
- les variables textuelles reçoivent une catégorie explicite `<MANQUANT>` ;
- des indicateurs binaires de valeurs manquantes sont ajoutés pour les trois
  colonnes les plus trouées.

Ainsi, le modèle peut utiliser les données complétées tout en sachant qu'une
valeur était absente à l'origine.

In [13]:
for colonne in trois_colonnes_plus_trouees:
    df[f"{colonne}_etait_manquant"] = (
        df[colonne]
        .isna()
        .astype(int)
    )

indicateurs_manquants = [
    f"{colonne}_etait_manquant"
    for colonne in trois_colonnes_plus_trouees
]

print("Indicateurs créés :")
print(indicateurs_manquants)

df[
    trois_colonnes_plus_trouees + indicateurs_manquants
].head()

Indicateurs créés :
['country_etait_manquant', 'state_etait_manquant', 'duration_hours_min_etait_manquant']


,country,state,duration_hours_min,country_etait_manquant,state_etait_manquant,duration_hours_min_etait_manquant
0,us,tx,45 minutes,0,0,0
1,<NA>,tx,1-2 hrs,1,0,0
2,gb,<NA>,20 seconds,0,1,0
3,us,tx,1/2 hour,0,0,0
4,us,hi,15 minutes,0,0,0


## 14. Découpage temporel identique à la phase 8

La phase 9 conserve une découpe chronologique afin que le modèle soit entraîné
sur le passé et évalué sur des observations plus récentes.

Les lignes sans `datetime` restent conservées dans le DataFrame global, mais ne
peuvent pas être utilisées dans ce découpage temporel.

In [14]:
df_temporel = df.loc[
    df["datetime"].notna()
].copy()

df_temporel = df_temporel.sort_values(
    "datetime"
).copy()

position_coupure = int(
    len(df_temporel) * (1 - TEST_SIZE)
)

date_coupure = df_temporel.iloc[
    position_coupure
]["datetime"]

df_train = df_temporel.loc[
    df_temporel["datetime"] < date_coupure
].copy()

df_test = df_temporel.loc[
    df_temporel["datetime"] >= date_coupure
].copy()

print(f"Date de coupure : {date_coupure}")
print(f"Train : {len(df_train)} relevés")
print(f"Test : {len(df_test)} relevés")

assert df_train["datetime"].max() < df_test["datetime"].min()

Date de coupure : 2012-01-17 18:00:00
Train : 69967 relevés
Test : 17492 relevés


## 15. Définition des variables du modèle

Les indicateurs de valeurs manquantes sont ajoutés aux variables numériques.

L'imputation médiane sera réalisée dans le pipeline. Comme le pipeline est
ajusté uniquement sur le train, les médianes sont calculées sans utiliser le
jeu de test.

In [15]:
features_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
] + indicateurs_manquants

features_modele = [
    "text_features_without_leakage",
] + features_numeriques

X_train = df_train[features_modele].copy()
X_test = df_test[features_modele].copy()

y_train = df_train["is_hoax"].copy()
y_test = df_test["is_hoax"].copy()

print("Variables numériques utilisées :")
print(features_numeriques)

print("\nDimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)

Variables numériques utilisées :
['duration_seconds', 'latitude', 'longitude', 'observation_year', 'observation_month', 'observation_hour', 'country_etait_manquant', 'state_etait_manquant', 'duration_hours_min_etait_manquant']

Dimensions de X_train : (69967, 10)
Dimensions de X_test : (17492, 10)


## 16. Construction du pipeline avec imputation

Le pipeline réalise les traitements dans le bon ordre :

1. apprentissage du vocabulaire TF-IDF sur le train ;
2. calcul des médianes sur le train ;
3. entraînement du modèle ;
4. application des mêmes transformations au test.

Les indicateurs de manque sont déjà présents dans les données et ne sont pas
effacés par l'imputation.

In [16]:
preprocessing = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=10_000,
                ngram_range=(1, 2),
            ),
            "text_features_without_leakage",
        ),
        (
            "numerique",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="median"),
                    ),
                ]
            ),
            features_numeriques,
        ),
    ]
)

modele_valeurs_manquantes = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

modele_valeurs_manquantes

,steps,"[('preprocessing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('texte', ...), ('numerique', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 17. Entraînement du modèle

Le modèle est entraîné sur les observations antérieures à la date de coupure.

In [17]:
modele_valeurs_manquantes.fit(
    X_train,
    y_train,
)

print("Entraînement terminé.")

Entraînement terminé.


c:\Users\serge\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 18. Évaluation sur les observations récentes

Les métriques sont calculées sur le jeu de test temporel. Elles permettent de
vérifier que le traitement des valeurs manquantes a bien été intégré sans
réintroduire de fuite de données.

In [18]:
y_pred = modele_valeurs_manquantes.predict(
    X_test
)

precision_phase9 = precision_score(
    y_test,
    y_pred,
    zero_division=0,
)

recall_phase9 = recall_score(
    y_test,
    y_pred,
    zero_division=0,
)

accuracy_phase9 = accuracy_score(
    y_test,
    y_pred,
)

print(f"Precision : {precision_phase9:.2%}")
print(f"Recall : {recall_phase9:.2%}")
print(f"Accuracy : {accuracy_phase9:.2%}")

Precision : 1.42%
Recall : 59.42%
Accuracy : 67.25%


## 19. Matrice de confusion

In [19]:
matrice_phase9 = confusion_matrix(
    y_test,
    y_pred,
)

df_matrice_phase9 = pd.DataFrame(
    matrice_phase9,
    index=[
        "Réel : non-canular",
        "Réel : canular",
    ],
    columns=[
        "Prédit : non-canular",
        "Prédit : canular",
    ],
)

df_matrice_phase9

,Prédit : non-canular,Prédit : canular
Réel : non-canular,11681,5673
Réel : canular,56,82


## 20. Export des résultats

Tous les résultats spécifiques à la phase 9 sont enregistrés dans le dossier :

```text
outputs/phase_9_valeurs_manquantes/
```

In [20]:
resume_manquants.to_csv(
    PHASE9_DIR / "resume_valeurs_manquantes.csv",
    index=True,
)

df_resultats_manquants_affichage.to_csv(
    PHASE9_DIR / "comparaison_canulars_valeurs_manquantes.csv",
    index=False,
)

df_matrice_phase9.to_csv(
    PHASE9_DIR / "matrice_confusion_phase9.csv",
    index=True,
)

resultats_modele_phase9 = pd.DataFrame(
    [
        {
            "modele": "Modèle avec indicateurs de valeurs manquantes",
            "date_coupure": date_coupure,
            "precision": precision_phase9,
            "recall": recall_phase9,
            "accuracy": accuracy_phase9,
            "nombre_train": len(df_train),
            "nombre_test": len(df_test),
            "colonnes_analysees": ", ".join(
                trois_colonnes_plus_trouees
            ),
            "indicateurs_manquants": ", ".join(
                indicateurs_manquants
            ),
        }
    ]
)

resultats_modele_phase9.to_csv(
    PHASE9_DIR / "resultats_modele_phase9.csv",
    index=False,
)

print("Fichiers exportés dans :")
print(PHASE9_DIR)

Fichiers exportés dans :
..\outputs\phase_9_valeurs_manquantes
